In [1]:

import os
import re


# ----------------------------
# Caption helpers
# ----------------------------

def underscores_to_spaces_in_caption(s: str) -> str:
    """Replace underscores with spaces in figure captions (human-readable)."""
    return s.replace("_", " ")


def escape_underscores_in_text(s: str) -> str:
    """
    Escape underscores in plain text, but NOT inside LaTeX commands.
    Useful for table cell text.
    """
    out = []
    i = 0
    while i < len(s):
        if s[i] == "\\":  # LaTeX command or escape, copy verbatim
            out.append(s[i])
            i += 1
            while i < len(s) and (s[i].isalpha() or s[i] == "*"):
                out.append(s[i])
                i += 1
        elif s[i] == "_":
            out.append(r"\_")
            i += 1
        else:
            out.append(s[i])
            i += 1
    return "".join(out)


# ----------------------------
# Companion caption .tex (optional)
# ----------------------------

def extract_caption_from_tex(tex_file: str):
    """
    Extract caption and label from a companion .tex file (if it exists).
    Returns (caption, label) or (None, None).
    """
    possible_paths = [
        tex_file,
        tex_file.replace(
            os.path.dirname(tex_file),
            os.path.join(os.path.dirname(tex_file), "latex_captions"),
        ),
    ]

    content = None
    for path in possible_paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
                break
        except (FileNotFoundError, IOError):
            continue

    if not content:
        return None, None

    # Caption (stop at \label if present)
    caption_match = re.search(r"\\caption\{(.+?)\}\s*(?:\\label|$)", content, re.DOTALL)
    caption = caption_match.group(1).strip() if caption_match else None
    if caption:
        caption = " ".join(caption.split())
        if caption.count("{") != caption.count("}"):
            print("Warning: Unbalanced braces in caption in:", tex_file)
            caption = None

    label_match = re.search(r"\\label\{(.+?)\}", content)
    label = label_match.group(1).strip() if label_match else None

    return caption, label


def generate_caption_from_filename(filename: str) -> str:
    """Generate a human-readable caption from filename (or companion .tex if available)."""
    tex_file = filename + ".tex"
    caption, _ = extract_caption_from_tex(tex_file)
    if caption:
        return caption

    base = os.path.splitext(os.path.basename(filename))[0]
    caption = base.replace("_", " ")
    caption = " ".join(word.capitalize() for word in caption.split())
    return caption


def generate_label_from_filename(filename: str) -> str:
    """Generate a LaTeX label from filename (or companion .tex if available)."""
    tex_file = filename + ".tex"
    _, label = extract_caption_from_tex(tex_file)
    if label:
        return label

    base = os.path.splitext(os.path.basename(filename))[0]
    safe = re.sub(r"[^a-zA-Z0-9_]", "_", base)
    return f"fig:{safe}"


# ----------------------------
# Table utilities
# ----------------------------

def count_columns_in_tabular(tex_path: str) -> int:
    """
    Count columns from the {<spec>} of \\begin{tabular}{<spec>}.
    Supports l/c/r/X and p{..}/m{..}/b{..}. Returns 0 if not found.
    """
    try:
        with open(tex_path, "r", encoding="utf-8") as f:
            s = f.read()
        m = re.search(r"\\begin\{tabular\}\{([^}]*)\}", s)
        if not m:
            return 0
        spec = m.group(1)
        return len(re.findall(r"(?:[lcrX]|p\{[^}]*\}|m\{[^}]*\}|b\{[^}]*\})", spec))
    except Exception:
        return 0


def convert_table_to_longtable(table_file: str) -> str:
    """
    Convert a tabular inside a .tex file to a longtable.

    - Robust row splitting: splits by \\\\ or \\tabularnewline
    - Removes \\hline / \\toprule / \\midrule / \\bottomrule from input body
    - Escapes underscores in table cell text (but not in LaTeX commands)
    - Skips tables that have no data rows
    - Uses unique defaults for caption/label based on filename if missing
    """
    try:
        with open(table_file, "r", encoding="utf-8") as f:
            content = f.read()

        base = os.path.splitext(os.path.basename(table_file))[0]
        default_caption = base.replace("_", " ")
        default_label = "tab:" + re.sub(r"[^a-zA-Z0-9]+", "_", base).strip("_").lower()

        caption_match = re.search(r"\\caption\{(.+?)\}", content, re.DOTALL)
        caption = caption_match.group(1).strip() if caption_match else default_caption

        label_match = re.search(r"\\label\{(.+?)\}", content)
        label = label_match.group(1).strip() if label_match else default_label

        tabular_match = re.search(
            r"\\begin\{tabular\}\{(?P<spec>[^}]*)\}(?P<body>.*?)\\end\{tabular\}",
            content,
            re.DOTALL,
        )
        if not tabular_match:
            # If can't parse, include as-is
            return f"\\input{{{table_file}}}\n\n"

        column_spec = tabular_match.group("spec").strip()
        body = tabular_match.group("body")

        # Normalize and clean
        body = body.replace("\\tabularnewline", "\\\\")
        body = re.sub(r"(?m)%.*$", "", body)  # strip comments

        # Remove rule commands from within rows; we'll add our own
        body = re.sub(r"\\hline\b", "", body)
        body = re.sub(r"\\(toprule|midrule|bottomrule)\b", "", body)

        # Split into rows by \\ (works even if multiple rows are on one physical line)
        rows = [r.strip() for r in re.split(r"\\\\", body) if r.strip()]
        if not rows:
            return f"% Skipped empty table: {table_file}\n\n"

        header_row = rows[0]
        data_rows = [r for r in rows[1:] if re.sub(r"\s+", "", r) != ""]

        # Skip tables with no data rows (prevents “header-only” scaffolds)
        if len(data_rows) == 0:
            return f"% Skipped header-only table: {table_file}\n\n"

        n_cols = header_row.count("&") + 1

        out = []
        out.append(f"\\begin{{longtable}}{{{column_spec}}}\n")
        out.append(f"\\caption{{{caption}}} \\label{{{label}}} \\\\\n")
        out.append("\\toprule\n")
        out.append(escape_underscores_in_text(header_row) + " \\\\\n")
        out.append("\\midrule\n")
        out.append("\\endfirsthead\n\n")

        out.append(f"\\multicolumn{{{n_cols}}}{{c}}{{\\textit{{Continued from previous page}}}} \\\\\n")
        out.append("\\toprule\n")
        out.append(escape_underscores_in_text(header_row) + " \\\\\n")
        out.append("\\midrule\n")
        out.append("\\endhead\n\n")

        out.append("\\midrule\n")
        out.append(f"\\multicolumn{{{n_cols}}}{{r}}{{\\textit{{Continued on next page}}}} \\\\\n")
        out.append("\\endfoot\n\n")

        out.append("\\bottomrule\n")
        out.append("\\endlastfoot\n\n")

        for r in data_rows:
            out.append(escape_underscores_in_text(r) + " \\\\\n")

        out.append("\\end{longtable}\n\n")
        return "".join(out)

    except Exception as e:
        print(f"Warning: Could not convert {table_file} to longtable: {e}")
        return f"\\input{{{table_file}}}\n\n"


# ----------------------------
# Main document generator
# ----------------------------

def create_latex_from_filelist(
    filelist_path,
    output_file="supplementary_material.tex",
    title="Supplementary Material",
    authors=None,
    affiliations=None,
    organize_by_prefix=True,
    landscape_if_cols_ge=6,   # <-- change threshold if desired
):
    if authors is None:
        authors = [("Author 1", [1]), ("Author 2", [1])]
    if affiliations is None:
        affiliations = ["Institution 1"]

    with open(filelist_path, "r", encoding="utf-8") as f:
        files = [line.strip() for line in f if line.strip() and not line.startswith("#")]

    figures = [f for f in files if f.endswith((".png", ".pdf", ".jpg", ".jpeg"))]
    tables = [
        f for f in files
        if f.endswith(".tex")
        and not any(f.endswith(ext + ".tex") for ext in [".png", ".pdf", ".jpg", ".jpeg"])
    ]

    # Group figures
    if organize_by_prefix:
        figure_groups = {}
        for fig in figures:
            basename = os.path.basename(fig)
            prefix = basename.split("_")[0] if "_" in basename else "other"
            figure_groups.setdefault(prefix, []).append(fig)
    else:
        figure_groups = {"all": figures}

    # Authors
    author_lines = []
    for author, affil_indices in authors:
        affil_string = ",".join(str(i) for i in affil_indices)
        author_lines.append(f"\\author[{affil_string}]{{{author}}}")
    author_string = "\n".join(author_lines)

    affiliation_lines = []
    for i, affil in enumerate(affiliations, 1):
        affiliation_lines.append(f"\\affil[{i}]{{{affil}}}")
    affiliation_string = "\n".join(affiliation_lines)

    # Preamble
    preamble = (
        r"""\documentclass[11pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{graphicx}
\DeclareGraphicsExtensions{.png,.pdf,.jpg,.jpeg}
\usepackage{booktabs}
\usepackage{float}
\usepackage{caption}
\usepackage{subcaption}
\usepackage{hyperref}
\usepackage{amsmath}
\usepackage{longtable}
\usepackage{authblk}
\usepackage{pdflscape}
\usepackage{array}

\title{"""
        + title
        + r"""}
"""
        + author_string
        + "\n"
        + affiliation_string
        + r"""
\date{\today}

\begin{document}
\maketitle

\tableofcontents
\clearpage

"""
    )

    content = [preamble]

    # Tables first
    if tables:
        content.append("\\section{Supplementary Tables}\n\n")

        for table_file in tables:
            ncols = count_columns_in_tabular(table_file)

            # Only landscape if the table is truly wide
            if ncols >= landscape_if_cols_ge:
                content.append("\\begin{landscape}\n")
                content.append("{\\footnotesize\n")
                content.append(convert_table_to_longtable(table_file))
                content.append("}\n")
                content.append("\\end{landscape}\n\n")
            else:
                content.append(convert_table_to_longtable(table_file))

        content.append("\\clearpage\n\n")

    # Figures by prefix group
    for prefix, figs in sorted(figure_groups.items()):
        section_title = "Figures" if prefix == "all" else f"{prefix.capitalize()} Figures"
        content.append(f"\\section{{{section_title}}}\n\n")

        base_groups = {}
        for fig in figs:
            basename = os.path.basename(fig)
            base_pattern = re.sub(
                r"_(Anglosphere|Germanic|Slavic|Latin|Turkic|Semetic|Indo-Iranian|EastAsia|SubSaharanAfrica|Other)_",
                "_",
                basename,
            )
            base_pattern = re.sub(r"_imp\.png$", "", base_pattern)
            base_groups.setdefault(base_pattern, []).append(fig)

        for base_pattern, group_figs in sorted(base_groups.items()):
            if len(group_figs) == 1:
                fig = group_figs[0]
                fig_path = fig.replace(".png.tex", ".png").replace(".pdf.tex", ".pdf")

                caption = generate_caption_from_filename(fig)
                caption = underscores_to_spaces_in_caption(caption)
                label = generate_label_from_filename(fig)

                content.append(
                    f"""\\begin{{figure}}[H]
    \\centering
    \\includegraphics[width=0.8\\textwidth]{{{fig_path}}}
    \\caption{{{caption}}}
    \\label{{{label}}}
\\end{{figure}}

"""
                )

            elif len(group_figs) <= 4:
                main_caption = generate_caption_from_filename(base_pattern)
                main_caption = underscores_to_spaces_in_caption(main_caption)
                label = generate_label_from_filename(base_pattern) + "_grid"

                content.append("\\begin{figure}[H]\n    \\centering\n")

                for i, fig in enumerate(group_figs):
                    subcaption = generate_caption_from_filename(fig)
                    subcaption = underscores_to_spaces_in_caption(subcaption)

                    # If you want compact subcaptions based on unique filename parts:
                    if "_" in os.path.basename(fig):
                        parts = os.path.basename(fig).replace(".png", "").split("_")
                        unique_parts = [p for p in parts if p not in base_pattern]
                        if unique_parts:
                            subcaption = " ".join(unique_parts).replace("_", " ").capitalize()

                    width = 0.45
                    content.append(f"    \\begin{{subfigure}}{{{width}\\textwidth}}\n")
                    content.append("        \\centering\n")
                    content.append(f"        \\includegraphics[width=\\textwidth]{{{fig}}}\n")
                    content.append(f"        \\caption{{{subcaption}}}\n")
                    content.append("    \\end{subfigure}\n")

                    if (i + 1) % 2 == 1 and i + 1 < len(group_figs):
                        content.append("    \\hfill\n")
                    elif i + 1 < len(group_figs):
                        content.append("    \\\\\n")

                content.append(f"    \\caption{{{main_caption}}}\n")
                content.append(f"    \\label{{{label}}}\n")
                content.append("\\end{figure}\n\n")

            else:
                for fig in group_figs:
                    caption = generate_caption_from_filename(fig)
                    caption = underscores_to_spaces_in_caption(caption)
                    label = generate_label_from_filename(fig)

                    content.append(
                        f"""\\begin{{figure}}[H]
    \\centering
    \\includegraphics[width=0.7\\textwidth]{{{fig}}}
    \\caption{{{caption}}}
    \\label{{{label}}}
\\end{{figure}}

"""
                    )

    content.append("\\end{document}\n")

    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(content)

    print(f"LaTeX file generated: {output_file}")
    print(f"Total figures: {len(figures)}")
    print(f"Total tables: {len(tables)}")
    print(f"Compile with: pdflatex {output_file}")


if __name__ == "__main__":
    affiliations = [
        "Tunis Business School-University of Tunis",
        "DTU Healthtech, Technical University of Denmark",
        "DTU Compute, Technical University of Denmark",
        "Université Laval",
    ]

    authors = [
        ("Meriem Boukhris", [1, 2]),
        ("Farah Benzarti", [1]),
        ("Sune Lehmann", [3]),
        ("Peter Wad Sackett", [2]),
        ("Montassar Ben Messaoud", [1]),
        ("Gabriel Renaud", [2, 4]),
    ]

    create_latex_from_filelist(
        filelist_path="figure_list.txt",
        output_file="supplementary_material.tex",
        title="Supplementary Material for: Language Clustering in World Values Survey",
        authors=authors,
        affiliations=affiliations,
        organize_by_prefix=True,
        landscape_if_cols_ge=6,  # adjust if you want more/less landscape tables
    )



LaTeX file generated: supplementary_material.tex
Total figures: 185
Total tables: 9
Compile with: pdflatex supplementary_material.tex
